# NYC 311 2025 — Data Recovery & Cleaning

This notebook documents the recovery, validation, cleaning, and preparation of the 2025 NYC 311 service request data for downstream SQL, geospatial, socioeconomic, and Power BI analysis.

**Analysis period:** January–December 2025

**Final dataset:** 3,655,040 service requests


## 1. Data Recovery

Recover and inspect the monthly NYC 311 source files used for the 2025 analysis.


In [1]:
from google.colab import files

uploaded = files.upload()


Saving NYC_311_2025_RAW.zip to NYC_311_2025_RAW (1).zip


In [2]:
import os

zip_path = "/content/NYC_311_2025_RAW (1).zip"

print("File exists:", os.path.exists(zip_path))

if os.path.exists(zip_path):
    print(f"File size: {os.path.getsize(zip_path) / (1024**3):.2f} GB")


File exists: True
File size: 0.18 GB


In [3]:
import zipfile

zip_path = "/content/NYC_311_2025_RAW (1).zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files_in_zip = z.namelist()

print("Files inside ZIP:")
print("=" * 60)

for file_name in files_in_zip:
    print(file_name)

print("\nTotal files:", len(files_in_zip))


Files inside ZIP:
nyc_311_2025_01.csv
nyc_311_2025_02.csv
nyc_311_2025_03.csv
nyc_311_2025_04.csv
nyc_311_2025_05.csv
nyc_311_2025_06.csv
nyc_311_2025_07.csv
nyc_311_2025_08.csv
nyc_311_2025_09.csv
nyc_311_2025_10.csv
nyc_311_2025_11.csv
nyc_311_2025_12.csv

Total files: 12


In [4]:
import zipfile
import os

zip_path = "/content/NYC_311_2025_RAW (1).zip"
extract_path = "/content/nyc_311_2025_raw"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction completed.")
print(f"Extracted to: {extract_path}")


Extraction completed.
Extracted to: /content/nyc_311_2025_raw


In [5]:
import os

extract_path = "/content/nyc_311_2025_raw"

csv_files = sorted(
    file_name
    for file_name in os.listdir(extract_path)
    if file_name.endswith(".csv")
)

print("Extracted CSV files:")
print("=" * 60)

for file_name in csv_files:
    file_path = os.path.join(extract_path, file_name)
    size_mb = os.path.getsize(file_path) / (1024**2)
    print(f"{file_name:<30} {size_mb:>8.2f} MB")

print("\nTotal CSV files:", len(csv_files))


Extracted CSV files:
nyc_311_2025_01.csv              132.90 MB
nyc_311_2025_02.csv               95.00 MB
nyc_311_2025_03.csv              101.30 MB
nyc_311_2025_04.csv               96.31 MB
nyc_311_2025_05.csv              101.89 MB
nyc_311_2025_06.csv              104.70 MB
nyc_311_2025_07.csv              108.70 MB
nyc_311_2025_08.csv              103.52 MB
nyc_311_2025_09.csv              102.45 MB
nyc_311_2025_10.csv              121.07 MB
nyc_311_2025_11.csv              147.39 MB
nyc_311_2025_12.csv              184.72 MB

Total CSV files: 12


In [6]:
import pandas as pd
import os

extract_path = "/content/nyc_311_2025_raw"

jan_path = os.path.join(
    extract_path,
    "nyc_311_2025_01.csv"
)

df_jan = pd.read_csv(jan_path)

print("January dataset loaded successfully.")
print("=" * 60)
print("Rows:", f"{len(df_jan):,}")
print("Columns:", len(df_jan.columns))

print("\nColumn names:")
print(df_jan.columns.tolist())

print("\nFirst 3 rows:")
display(df_jan.head(3))


/tmp/ipykernel_759/312778271.py:11: DtypeWarning: Columns (7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df_jan = pd.read_csv(jan_path)


January dataset loaded successfully.
Rows: 348,180
Columns: 14

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'incident_zip', 'status', 'resolution_description', 'community_board', 'borough', 'latitude', 'longitude']

First 3 rows:


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,incident_zip,status,resolution_description,community_board,borough,latitude,longitude
0,63572086,2025-01-01T00:51:02.000,2025-03-02T00:51:37.000,DOHMH,Department of Health and Mental Hygiene,Smoking or Vaping,Allowed in Smoke Free Area,10475,Closed,The NYC Health Department has responded to you...,10 BRONX,BRONX,40.875189,-73.82899057253027
1,63572092,2025-01-01T00:54:26.000,2025-01-01T01:47:36.000,NYPD,New York City Police Department,Illegal Parking,Parking Permit Improper Use,11373,Closed,The Police Department responded to the complai...,04 QUEENS,QUEENS,40.743725,-73.87311388337658
2,63572113,2025-01-01T01:14:30.000,2025-01-01T04:38:47.000,NYPD,New York City Police Department,Blocked Driveway,No Access,11208,Closed,The Police Department responded and upon arriv...,05 BROOKLYN,BROOKLYN,40.673220,-73.87800662913914


## 2. Monthly Data Validation

Validate monthly record counts and confirm that the recovered files match the expected 2025 dataset coverage.


In [7]:
import pandas as pd
import os

extract_path = "/content/nyc_311_2025_raw"

expected_counts = {
    "01": 348180,
    "02": 255364,
    "03": 281221,
    "04": 272549,
    "05": 295058,
    "06": 306442,
    "07": 315883,
    "08": 304039,
    "09": 302684,
    "10": 336612,
    "11": 304905,
    "12": 332103
}

validation_results = []

for month in range(1, 13):
    month_str = f"{month:02d}"
    file_name = f"nyc_311_2025_{month_str}.csv"
    file_path = os.path.join(extract_path, file_name)

    temp_df = pd.read_csv(file_path)

    validation_results.append({
        "Month": month_str,
        "Rows": len(temp_df),
        "Expected": expected_counts[month_str],
        "Columns": len(temp_df.columns),
        "Rows Match": len(temp_df) == expected_counts[month_str]
    })

    del temp_df

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("\n" + "=" * 60)
print("TOTAL ROWS:", f"{validation_df['Rows'].sum():,}")
print("EXPECTED TOTAL:", f"{sum(expected_counts.values()):,}")
print(
    "PASS - All monthly row counts match"
    if validation_df["Rows Match"].all()
    else "FAIL - One or more monthly counts do not match"
)


/tmp/ipykernel_759/1653177308.py:28: DtypeWarning: Columns (7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)
/tmp/ipykernel_759/1653177308.py:28: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)
/tmp/ipykernel_759/1653177308.py:28: DtypeWarning: Columns (7,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)
/tmp/ipykernel_759/1653177308.py:28: DtypeWarning: Columns (7,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)
/tmp/ipykernel_759/1653177308.py:28: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


,Month,Rows,Expected,Columns,Rows Match
0,01,348180,348180,14,True
1,02,255364,255364,14,True
2,03,281221,281221,14,True
3,04,272549,272549,14,True
4,05,295058,295058,14,True
5,06,306442,306442,14,True
6,07,315883,315883,14,True
7,08,304039,304039,14,True
8,09,302684,302684,14,True
9,10,336612,336612,14,True



TOTAL ROWS: 3,655,040
EXPECTED TOTAL: 3,655,040
PASS - All monthly row counts match


In [8]:
import pandas as pd
import os

extract_path = "/content/nyc_311_2025_raw"

monthly_dfs = []

for month in range(1, 13):
    month_str = f"{month:02d}"
    file_name = f"nyc_311_2025_{month_str}.csv"
    file_path = os.path.join(extract_path, file_name)

    temp_df = pd.read_csv(file_path)
    temp_df["source_month"] = month_str

    monthly_dfs.append(temp_df)

    print(
        f"Loaded {file_name}: "
        f"{len(temp_df):,} rows"
    )

df_raw = pd.concat(
    monthly_dfs,
    ignore_index=True
)

del monthly_dfs

print("\n" + "=" * 60)
print("COMBINED RAW DATASET")
print("=" * 60)
print("Rows:", f"{len(df_raw):,}")
print("Columns:", len(df_raw.columns))
print("Expected rows: 3,655,040")


/tmp/ipykernel_759/2224239819.py:13: DtypeWarning: Columns (7,13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


Loaded nyc_311_2025_01.csv: 348,180 rows
Loaded nyc_311_2025_02.csv: 255,364 rows
Loaded nyc_311_2025_03.csv: 281,221 rows
Loaded nyc_311_2025_04.csv: 272,549 rows


/tmp/ipykernel_759/2224239819.py:13: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


Loaded nyc_311_2025_05.csv: 295,058 rows


/tmp/ipykernel_759/2224239819.py:13: DtypeWarning: Columns (7,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


Loaded nyc_311_2025_06.csv: 306,442 rows
Loaded nyc_311_2025_07.csv: 315,883 rows


/tmp/ipykernel_759/2224239819.py:13: DtypeWarning: Columns (7,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


Loaded nyc_311_2025_08.csv: 304,039 rows
Loaded nyc_311_2025_09.csv: 302,684 rows
Loaded nyc_311_2025_10.csv: 336,612 rows
Loaded nyc_311_2025_11.csv: 304,905 rows


/tmp/ipykernel_759/2224239819.py:13: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  temp_df = pd.read_csv(file_path)


Loaded nyc_311_2025_12.csv: 332,103 rows

COMBINED RAW DATASET
Rows: 3,655,040
Columns: 15
Expected rows: 3,655,040


## 3. Combined Dataset Validation

Combine the monthly files and validate record counts, service request IDs, month coverage, and date ranges.


In [9]:
print("COMBINED RAW DATA VALIDATION")
print("=" * 70)

print("\n1. SHAPE")
print("-" * 70)
print("Rows:", f"{len(df_raw):,}")
print("Columns:", len(df_raw.columns))

print("\n2. UNIQUE SERVICE REQUEST IDs")
print("-" * 70)

unique_ids = df_raw["unique_key"].nunique()
duplicate_ids = df_raw["unique_key"].duplicated().sum()

print("Unique IDs:", f"{unique_ids:,}")
print("Duplicate IDs:", f"{duplicate_ids:,}")

print(
    "PASS - All service request IDs are unique"
    if duplicate_ids == 0
    else "FAIL - Duplicate service request IDs detected"
)

print("\n3. MONTH DISTRIBUTION")
print("-" * 70)

month_counts = (
    df_raw["source_month"]
    .value_counts()
    .sort_index()
)

display(month_counts.to_frame("Records"))

print("\n4. DATE RANGE")
print("-" * 70)

created_dates = pd.to_datetime(
    df_raw["created_date"],
    errors="coerce"
)

print("Minimum created date:", created_dates.min())
print("Maximum created date:", created_dates.max())

print("\n" + "=" * 70)
print("RAW DATA VALIDATION COMPLETE")
print("=" * 70)


COMBINED RAW DATA VALIDATION

1. SHAPE
----------------------------------------------------------------------
Rows: 3,655,040
Columns: 15

2. UNIQUE SERVICE REQUEST IDs
----------------------------------------------------------------------
Unique IDs: 3,655,040
Duplicate IDs: 0
PASS - All service request IDs are unique

3. MONTH DISTRIBUTION
----------------------------------------------------------------------


,Records
source_month,
01,348180
02,255364
03,281221
04,272549
05,295058
06,306442
07,315883
08,304039
09,302684



4. DATE RANGE
----------------------------------------------------------------------
Minimum created date: 2025-01-01 00:00:12
Maximum created date: 2025-12-31 23:59:28

RAW DATA VALIDATION COMPLETE


## 4. Data Profiling

Create the profiling dataframe and inspect the structure and data types of the recovered dataset.


In [10]:
# Preserve the recovered raw dataset
df_profile = df_raw.copy()

print("Profiling dataframe created.")
print("=" * 70)
print("Rows:", f"{len(df_profile):,}")
print("Columns:", len(df_profile.columns))

print("\nColumn names:")
print(df_profile.columns.tolist())

print("\nData types:")
print(df_profile.dtypes)


Profiling dataframe created.
Rows: 3,655,040
Columns: 15

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'incident_zip', 'status', 'resolution_description', 'community_board', 'borough', 'latitude', 'longitude', 'source_month']

Data types:
unique_key                 int64
created_date              object
closed_date               object
agency                    object
agency_name               object
complaint_type            object
descriptor                object
incident_zip              object
status                    object
resolution_description    object
community_board           object
borough                   object
latitude                  object
longitude                 object
source_month              object
dtype: object


## 5. Date Standardization

Convert request creation and closure timestamps into standardized datetime fields and assess date quality.


In [11]:
# Create cleaning dataframe
df_clean = df_profile.copy()

# Convert date columns to datetime
df_clean["created_datetime"] = pd.to_datetime(
    df_clean["created_date"],
    errors="coerce"
)

df_clean["closed_datetime"] = pd.to_datetime(
    df_clean["closed_date"],
    errors="coerce"
)

print("DATE STANDARDIZATION")
print("=" * 70)

print("\nCreated date:")
print("Valid:", f"{df_clean['created_datetime'].notna().sum():,}")
print("Invalid/Missing:", f"{df_clean['created_datetime'].isna().sum():,}")
print("Minimum:", df_clean["created_datetime"].min())
print("Maximum:", df_clean["created_datetime"].max())

print("\nClosed date:")
print("Valid:", f"{df_clean['closed_datetime'].notna().sum():,}")
print("Invalid/Missing:", f"{df_clean['closed_datetime'].notna().eq(False).sum():,}")
print("Minimum:", df_clean["closed_datetime"].min())
print("Maximum:", df_clean["closed_datetime"].max())


DATE STANDARDIZATION

Created date:
Valid: 3,655,040
Invalid/Missing: 0
Minimum: 2025-01-01 00:00:12
Maximum: 2025-12-31 23:59:28

Closed date:
Valid: 3,526,004
Invalid/Missing: 129,036
Minimum: 2024-10-21 11:06:00
Maximum: 2026-09-24 16:02:20


## 6. Resolution Time Validation

Calculate resolution duration and classify valid, missing, and negative resolution records.


In [12]:
# Calculate raw resolution duration first
raw_resolution_hours = (
    df_clean["closed_datetime"] - df_clean["created_datetime"]
).dt.total_seconds() / 3600

# Identify negative durations
negative_duration = raw_resolution_hours < 0

# Identify valid resolution durations
valid_duration = (
    df_clean["created_datetime"].notna()
    & df_clean["closed_datetime"].notna()
    & (raw_resolution_hours >= 0)
)

# Keep only valid durations
df_clean["resolution_hours"] = raw_resolution_hours.where(
    valid_duration
)

df_clean["resolution_days"] = (
    df_clean["resolution_hours"] / 24
)

# Classify resolution quality
df_clean["resolution_quality"] = "Valid"

df_clean.loc[
    df_clean["closed_datetime"].isna(),
    "resolution_quality"
] = "Missing / Invalid Closed Date"

df_clean.loc[
    negative_duration,
    "resolution_quality"
] = "Negative Duration"

print("RESOLUTION TIME VALIDATION")
print("=" * 70)

print("\nResolution quality:")
display(
    df_clean["resolution_quality"]
    .value_counts()
    .to_frame("Records")
)

print("\nResolution hours:")
print(
    df_clean["resolution_hours"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

print("\nNegative resolution hours remaining:",
      df_clean["resolution_hours"].dropna().lt(0).sum())


RESOLUTION TIME VALIDATION

Resolution quality:


,Records
resolution_quality,
Valid,3525111
Missing / Invalid Closed Date,129036
Negative Duration,893



Resolution hours:
count    3.525111e+06
mean     2.346127e+02
std      8.843923e+02
min      0.000000e+00
25%      1.046667e+00
50%      6.683333e+00
75%      6.735556e+01
95%      1.233460e+03
99%      4.879780e+03
max      1.498477e+04
Name: resolution_hours, dtype: float64

Negative resolution hours remaining: 0


## 7. Agency Standardization

Standardize agency identifiers into consistent agency names for downstream analysis.


In [13]:
# Standardize agency identifiers
agency_series = (
    df_clean["agency"]
    .astype("string")
    .str.strip()
    .str.upper()
)

agency_mapping = {
    "NYPD": "New York City Police Department",
    "NEW YORK CITY POLICE DEPARTMENT": "New York City Police Department",

    "HPD": "Department of Housing Preservation and Development",
    "DEPARTMENT OF HOUSING PRESERVATION AND DEVELOPMENT":
        "Department of Housing Preservation and Development",

    "DSNY": "Department of Sanitation",
    "DEPARTMENT OF SANITATION": "Department of Sanitation",

    "DOT": "Department of Transportation",
    "DEPARTMENT OF TRANSPORTATION": "Department of Transportation",

    "DEP": "Department of Environmental Protection",
    "DEPARTMENT OF ENVIRONMENTAL PROTECTION":
        "Department of Environmental Protection",

    "DPR": "Department of Parks and Recreation",
    "DEPARTMENT OF PARKS AND RECREATION":
        "Department of Parks and Recreation",

    "DOB": "Department of Buildings",
    "DEPARTMENT OF BUILDINGS": "Department of Buildings",

    "DOHMH": "Department of Health and Mental Hygiene",
    "DEPARTMENT OF HEALTH AND MENTAL HYGIENE":
        "Department of Health and Mental Hygiene",

    "DHS": "Department of Homeless Services",
    "DEPARTMENT OF HOMELESS SERVICES":
        "Department of Homeless Services",

    "TLC": "Taxi and Limousine Commission",
    "TAXI AND LIMOUSINE COMMISSION":
        "Taxi and Limousine Commission",

    "EDC": "Economic Development Corporation",
    "ECONOMIC DEVELOPMENT CORPORATION":
        "Economic Development Corporation",

    "DCWP": "Department of Consumer and Worker Protection",
    "DEPARTMENT OF CONSUMER AND WORKER PROTECTION":
        "Department of Consumer and Worker Protection",

    "DOE": "Department of Education",
    "DEPARTMENT OF EDUCATION":
        "Department of Education",

    "OOS": "Office of the Sheriff",

    "OTI": "Office of Technology and Innovation",
    "OFFICE OF TECHNOLOGY AND INNOVATION":
        "Office of Technology and Innovation"
}

df_clean["agency_clean"] = agency_series.map(agency_mapping)

print("AGENCY STANDARDIZATION")
print("=" * 70)

print("Unique raw agency values:",
      df_clean["agency"].nunique(dropna=True))

print("Unique standardized agencies:",
      df_clean["agency_clean"].nunique(dropna=True))

print("Missing standardized agency:",
      df_clean["agency_clean"].isna().sum())

print("\nAgency distribution:")
display(
    df_clean["agency_clean"]
    .value_counts()
    .to_frame("Records")
)


AGENCY STANDARDIZATION
Unique raw agency values: 29
Unique standardized agencies: 15
Missing standardized agency: 0

Agency distribution:


,Records
agency_clean,
New York City Police Department,1717594
Department of Housing Preservation and Development,774535
Department of Sanitation,326780
Department of Transportation,212103
Department of Environmental Protection,197143
Department of Parks and Recreation,110855
Department of Buildings,103066
Department of Health and Mental Hygiene,83396
Department of Homeless Services,51449


## 8. Status Standardization

Standardize operational status values and identify unrecognized or missing records.


In [14]:
# Standardize operational status values
status_series = (
    df_clean["status"]
    .astype("string")
    .str.strip()
)

known_statuses = [
    "Closed",
    "Open",
    "In Progress",
    "Pending",
    "Assigned",
    "Started"
]

df_clean["status_clean"] = status_series.where(
    status_series.isin(known_statuses),
    pd.NA
)

print("STATUS STANDARDIZATION")
print("=" * 70)

print("Recognized status records:",
      f"{df_clean['status_clean'].notna().sum():,}")

print("Unrecognized / missing:",
      f"{df_clean['status_clean'].isna().sum():,}")

print("\nStatus distribution:")
display(
    df_clean["status_clean"]
    .value_counts(dropna=False)
    .to_frame("Records")
)


STATUS STANDARDIZATION
Recognized status records: 3,504,877
Unrecognized / missing: 150,163

Status distribution:


,Records
status_clean,
Closed,3453793
<NA>,150163
In Progress,36856
Open,5802
Pending,4119
Assigned,3213
Started,1094


## 9. Complaint Type Cleaning

Clean complaint type values and review the distribution of service request categories.


In [15]:
# Clean complaint type
df_clean["complaint_type_clean"] = (
    df_clean["complaint_type"]
    .astype("string")
    .str.strip()
)

# Convert blank strings to missing
df_clean.loc[
    df_clean["complaint_type_clean"].eq(""),
    "complaint_type_clean"
] = pd.NA

print("COMPLAINT TYPE VALIDATION")
print("=" * 70)

print(
    "Unique complaint types:",
    df_clean["complaint_type_clean"].nunique(dropna=True)
)

print(
    "Missing complaint types:",
    f"{df_clean['complaint_type_clean'].isna().sum():,}"
)

print("\nTop 10 complaint types:")
display(
    df_clean["complaint_type_clean"]
    .value_counts()
    .head(10)
    .to_frame("Records")
)


COMPLAINT TYPE VALIDATION
Unique complaint types: 874
Missing complaint types: 2,172

Top 10 complaint types:


,Records
complaint_type_clean,
Illegal Parking,564577
Noise - Residential,456599
HEAT/HOT WATER,315265
Blocked Driveway,169126
Noise - Street/Sidewalk,168021
UNSANITARY CONDITION,115996
Water System,75994
PLUMBING,68741
Street Condition,68208


## 10. Geographic Data Cleaning

Clean ZIP codes, borough information, and geographic identifiers used for spatial analysis.


In [16]:
# ============================================================
# GEOGRAPHIC FIELD CLEANING
# ============================================================

# 1. Clean ZIP codes
df_clean["incident_zip_clean"] = (
    df_clean["incident_zip"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Keep only valid 5-digit ZIP codes
df_clean.loc[
    ~df_clean["incident_zip_clean"].str.match(
        r"^\d{5}$",
        na=False
    ),
    "incident_zip_clean"
] = pd.NA

# 2. Recover borough information from community board
borough_from_board = (
    df_clean["community_board"]
    .astype("string")
    .str.upper()
    .str.extract(
        r"(BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)",
        expand=False
    )
)

standard_boroughs = [
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
]

# 3. Clean borough
df_clean["borough_clean"] = df_clean["borough"]

invalid_borough = ~df_clean["borough"].isin(
    standard_boroughs
)

# Recover borough only where the original value is invalid
df_clean.loc[
    invalid_borough & borough_from_board.notna(),
    "borough_clean"
] = borough_from_board[
    invalid_borough & borough_from_board.notna()
]

# Anything still outside the five NYC boroughs becomes missing
df_clean.loc[
    ~df_clean["borough_clean"].isin(standard_boroughs),
    "borough_clean"
] = pd.NA

# 4. Clean latitude and longitude
df_clean["latitude_clean"] = pd.to_numeric(
    df_clean["latitude"],
    errors="coerce"
)

df_clean["longitude_clean"] = pd.to_numeric(
    df_clean["longitude"],
    errors="coerce"
)

# Keep only plausible NYC coordinate ranges
df_clean.loc[
    ~df_clean["latitude_clean"].between(40.4, 41.0),
    "latitude_clean"
] = pd.NA

df_clean.loc[
    ~df_clean["longitude_clean"].between(-74.3, -73.6),
    "longitude_clean"
] = pd.NA

# ============================================================
# VALIDATION
# ============================================================

print("GEOGRAPHIC FIELD CLEANING")
print("=" * 70)

print(
    "Valid ZIP:",
    f"{df_clean['incident_zip_clean'].notna().sum():,}"
)

print(
    "Valid Borough:",
    f"{df_clean['borough_clean'].notna().sum():,}"
)

print(
    "Valid Latitude:",
    f"{df_clean['latitude_clean'].notna().sum():,}"
)

print(
    "Valid Longitude:",
    f"{df_clean['longitude_clean'].notna().sum():,}"
)


GEOGRAPHIC FIELD CLEANING
Valid ZIP: 3,476,461
Valid Borough: 3,396,220
Valid Latitude: 3,303,394
Valid Longitude: 3,303,394


## 11. Community Board Cleaning

Standardize community board identifiers and extract board numbers and associated boroughs.


In [17]:
# ============================================================
# COMMUNITY BOARD CLEANING
# ============================================================

cb = (
    df_clean["community_board"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Recognize official NYC community board format
standard_cb = cb.str.extract(
    r"^((?:0[1-9]|1[0-8])\s+"
    r"(?:BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND))$",
    expand=False
)

df_clean["community_board_clean"] = standard_cb

# Community Board number
df_clean["community_board_number"] = (
    df_clean["community_board_clean"]
    .str.extract(r"^(\d{2})\s+", expand=False)
)

# Borough associated with the validated board
df_clean["community_board_borough"] = (
    df_clean["community_board_clean"]
    .str.extract(
        r"\s+(BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$",
        expand=False
    )
)

# Recover borough from malformed community-board values
borough_from_cb = cb.str.extract(
    r"(BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)",
    expand=False
)

df_clean["community_board_borough_recovered"] = borough_from_cb

# Classify community-board quality
df_clean["community_board_quality"] = "Missing / Invalid"

df_clean.loc[
    df_clean["community_board_clean"].notna(),
    "community_board_quality"
] = "Valid Community Board"

df_clean.loc[
    df_clean["community_board_clean"].isna()
    & df_clean["community_board_borough_recovered"].notna(),
    "community_board_quality"
] = "Borough Recovered Only"

# ============================================================
# VALIDATION
# ============================================================

print("COMMUNITY BOARD VALIDATION")
print("=" * 70)

print("\nCommunity board quality:")
display(
    df_clean["community_board_quality"]
    .value_counts()
    .to_frame("Records")
)

print(
    "\nValid Community Boards:",
    f"{df_clean['community_board_clean'].notna().sum():,}"
)

print(
    "Unique valid Community Boards:",
    df_clean["community_board_clean"].nunique()
)

print("\nCommunity Board borough distribution:")
display(
    df_clean["community_board_borough"]
    .value_counts(dropna=False)
    .to_frame("Records")
)


COMMUNITY BOARD VALIDATION

Community board quality:


,Records
community_board_quality,
Valid Community Board,3246595
Missing / Invalid,258820
Borough Recovered Only,149625



Valid Community Boards: 3,246,595
Unique valid Community Boards: 59

Community Board borough distribution:


,Records
community_board_borough,
BROOKLYN,964915
QUEENS,775361
BRONX,750403
MANHATTAN,636721
<NA>,408445
STATEN ISLAND,119195


## 12. Community Board Validation

Validate cleaned community boards against the official NYC community board structure.


In [18]:
# ============================================================
# VALIDATE COMMUNITY BOARDS AGAINST OFFICIAL BOARD SET
# ============================================================

official_community_boards = set()

# Bronx: 01-12
for i in range(1, 13):
    official_community_boards.add(f"{i:02d} BRONX")

# Brooklyn: 01-18
for i in range(1, 19):
    official_community_boards.add(f"{i:02d} BROOKLYN")

# Manhattan: 01-12
for i in range(1, 13):
    official_community_boards.add(f"{i:02d} MANHATTAN")

# Queens: 01-14
for i in range(1, 15):
    official_community_boards.add(f"{i:02d} QUEENS")

# Staten Island: 01-03
for i in range(1, 4):
    official_community_boards.add(f"{i:02d} STATEN ISLAND")

# Validate each cleaned board
df_clean["community_board_validated"] = (
    df_clean["community_board_clean"]
    .where(
        df_clean["community_board_clean"].isin(
            official_community_boards
        )
    )
)

# ============================================================
# VALIDATION RESULTS
# ============================================================

invalid_boards = (
    df_clean["community_board_validated"].notna()
    & ~df_clean["community_board_validated"].isin(
        official_community_boards
    )
).sum()

print("COMMUNITY BOARD OFFICIAL SET VALIDATION")
print("=" * 70)

print(
    "Official board combinations:",
    len(official_community_boards)
)

print(
    "Valid community board records:",
    f"{df_clean['community_board_validated'].notna().sum():,}"
)

print(
    "Invalid validated-board records:",
    f"{invalid_boards:,}"
)

print(
    "Unique validated boards:",
    df_clean["community_board_validated"].nunique()
)

print(
    "\nPASS - All validated Community Boards belong to the official 59-board set."
    if invalid_boards == 0
    else "\nFAIL - Unexpected Community Board values detected."
)


COMMUNITY BOARD OFFICIAL SET VALIDATION
Official board combinations: 59
Valid community board records: 3,246,595
Invalid validated-board records: 0
Unique validated boards: 59

PASS - All validated Community Boards belong to the official 59-board set.


## 13. Geographic Quality Classification

Classify records according to the geographic information available for each service request.


In [19]:
# ============================================================
# FINAL GEOGRAPHIC QUALITY CLASSIFICATION
# ============================================================

df_clean["geo_quality_final"] = "No Usable Geography"

valid_borough = df_clean["borough_clean"].notna()

valid_board = (
    df_clean["community_board_validated"].notna()
)

valid_coordinates = (
    df_clean["latitude_clean"].notna()
    & df_clean["longitude_clean"].notna()
)

valid_zip = df_clean["incident_zip_clean"].notna()

# Coordinates + Borough + Community Board
df_clean.loc[
    valid_coordinates
    & valid_borough
    & valid_board,
    "geo_quality_final"
] = "Coordinates + Borough + Community Board"

# Coordinates + Borough
df_clean.loc[
    valid_coordinates
    & valid_borough
    & ~valid_board,
    "geo_quality_final"
] = "Coordinates + Borough"

# Borough + Community Board
df_clean.loc[
    ~valid_coordinates
    & valid_borough
    & valid_board,
    "geo_quality_final"
] = "Borough + Community Board"

# Borough + ZIP
df_clean.loc[
    ~valid_coordinates
    & valid_borough
    & ~valid_board
    & valid_zip,
    "geo_quality_final"
] = "Borough + ZIP"

# Borough Only
df_clean.loc[
    ~valid_coordinates
    & valid_borough
    & ~valid_board
    & ~valid_zip,
    "geo_quality_final"
] = "Borough Only"

# Coordinates Only
df_clean.loc[
    valid_coordinates
    & ~valid_borough,
    "geo_quality_final"
] = "Coordinates Only"

# ============================================================
# VALIDATION
# ============================================================

print("FINAL GEOGRAPHIC QUALITY")
print("=" * 70)

geo_summary = (
    df_clean["geo_quality_final"]
    .value_counts()
    .to_frame("Records")
)

geo_summary["Percentage"] = (
    geo_summary["Records"]
    / len(df_clean)
    * 100
)

display(geo_summary)

print("\nTotal classified records:",
      f"{geo_summary['Records'].sum():,}")

print(
    "\nPASS - Every record has exactly one geographic quality category."
    if geo_summary["Records"].sum() == len(df_clean)
    else "\nFAIL - Geographic classification count mismatch."
)


FINAL GEOGRAPHIC QUALITY


,Records,Percentage
geo_quality_final,,
Coordinates + Borough + Community Board,3226135,88.265382
No Usable Geography,258663,7.076886
Coordinates + Borough,77102,2.109471
Borough Only,61248,1.675714
Borough + Community Board,20460,0.559775
Borough + ZIP,11275,0.308478
Coordinates Only,157,0.004295



Total classified records: 3,655,040

PASS - Every record has exactly one geographic quality category.


## 14. Final Analytical Dataset

Select the validated analytical fields and rename them using consistent project-level column names.


In [20]:
# ============================================================
# CREATE FINAL ANALYTICAL DATASET
# ============================================================

final_columns = [
    "unique_key",
    "created_datetime",
    "closed_datetime",
    "resolution_hours",
    "resolution_days",
    "resolution_quality",
    "agency_clean",
    "complaint_type_clean",
    "descriptor",
    "status_clean",
    "resolution_description",
    "incident_zip_clean",
    "borough_clean",
    "community_board_validated",
    "community_board_number",
    "community_board_borough",
    "latitude_clean",
    "longitude_clean",
    "geo_quality_final"
]

df_final = df_clean[final_columns].copy()

# Rename columns to clean analytical names
df_final = df_final.rename(columns={
    "unique_key": "service_request_id",
    "created_datetime": "created_at",
    "closed_datetime": "closed_at",
    "agency_clean": "agency",
    "complaint_type_clean": "complaint_type",
    "status_clean": "status",
    "incident_zip_clean": "incident_zip",
    "borough_clean": "borough",
    "community_board_validated": "community_board",
    "latitude_clean": "latitude",
    "longitude_clean": "longitude"
})

print("FINAL ANALYTICAL DATASET")
print("=" * 70)

print("Rows:", f"{len(df_final):,}")
print("Columns:", len(df_final.columns))

print("\nColumn names:")
print(df_final.columns.tolist())


FINAL ANALYTICAL DATASET
Rows: 3,655,040
Columns: 19

Column names:
['service_request_id', 'created_at', 'closed_at', 'resolution_hours', 'resolution_days', 'resolution_quality', 'agency', 'complaint_type', 'descriptor', 'status', 'resolution_description', 'incident_zip', 'borough', 'community_board', 'community_board_number', 'community_board_borough', 'latitude', 'longitude', 'geo_quality_final']


## 15. Final Validation

Run final structural, identifier, geographic, and data-quality checks before downstream analysis.


In [21]:
print("FINAL ANALYTICAL DATASET VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. COLUMN CHECK
# ------------------------------------------------------------

required_columns = [
    "service_request_id",
    "created_at",
    "closed_at",
    "resolution_hours",
    "resolution_days",
    "resolution_quality",
    "agency",
    "complaint_type",
    "descriptor",
    "status",
    "resolution_description",
    "incident_zip",
    "borough",
    "community_board",
    "community_board_number",
    "community_board_borough",
    "latitude",
    "longitude",
    "geo_quality_final"
]

missing_columns = [
    col for col in required_columns
    if col not in df_final.columns
]

print("\n1. COLUMN CHECK")
print("-" * 80)

print(
    "PASS - All required columns are present"
    if len(missing_columns) == 0
    else f"FAIL - Missing columns: {missing_columns}"
)

# ------------------------------------------------------------
# 2. ID VALIDATION
# ------------------------------------------------------------

print("\n2. ID VALIDATION")
print("-" * 80)

id_unique = df_final["service_request_id"].is_unique
id_missing = df_final["service_request_id"].isna().sum()

print("Unique IDs:", id_unique)
print("Missing IDs:", id_missing)

print(
    "PASS - Service request IDs are unique and complete"
    if id_unique and id_missing == 0
    else "FAIL - ID validation issue"
)

# ------------------------------------------------------------
# 3. DATE VALIDATION
# ------------------------------------------------------------

print("\n3. DATE VALIDATION")
print("-" * 80)

created_invalid = df_final["created_at"].isna().sum()

print("Missing created_at:", created_invalid)

print(
    "PASS - Created dates are complete"
    if created_invalid == 0
    else "FAIL - Created dates contain missing values"
)

# ------------------------------------------------------------
# 4. RESOLUTION VALIDATION
# ------------------------------------------------------------

print("\n4. RESOLUTION VALIDATION")
print("-" * 80)

negative_hours = (
    df_final["resolution_hours"]
    .dropna()
    .lt(0)
    .sum()
)

valid_hours = (
    df_final["resolution_quality"] == "Valid"
).sum()

negative_quality = (
    df_final["resolution_quality"]
    == "Negative Duration"
).sum()

print("Negative resolution hours:", negative_hours)
print("Valid resolution records:", f"{valid_hours:,}")
print("Negative duration records:", f"{negative_quality:,}")

print(
    "PASS - No negative values remain in resolution_hours"
    if negative_hours == 0
    else "FAIL - Negative resolution values detected"
)

# ------------------------------------------------------------
# 5. COORDINATE VALIDATION
# ------------------------------------------------------------

print("\n5. COORDINATE VALIDATION")
print("-" * 80)

invalid_latitude = (
    df_final["latitude"].notna()
    & ~df_final["latitude"].between(40.4, 41.0)
).sum()

invalid_longitude = (
    df_final["longitude"].notna()
    & ~df_final["longitude"].between(-74.3, -73.6)
).sum()

print("Invalid latitude values:", invalid_latitude)
print("Invalid longitude values:", invalid_longitude)

print(
    "PASS - Coordinate ranges are valid"
    if invalid_latitude == 0
    and invalid_longitude == 0
    else "FAIL - Invalid coordinate values detected"
)

# ------------------------------------------------------------
# 6. BOROUGH VALIDATION
# ------------------------------------------------------------

print("\n6. BOROUGH VALIDATION")
print("-" * 80)

valid_boroughs = {
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
}

invalid_boroughs = df_final.loc[
    df_final["borough"].notna()
    & ~df_final["borough"].isin(valid_boroughs),
    "borough"
].unique()

print("Invalid borough values:", len(invalid_boroughs))

print(
    "PASS - Borough values are standardized"
    if len(invalid_boroughs) == 0
    else "FAIL - Invalid borough values detected"
)

# ------------------------------------------------------------
# 7. COMMUNITY BOARD VALIDATION
# ------------------------------------------------------------

print("\n7. COMMUNITY BOARD VALIDATION")
print("-" * 80)

invalid_community_boards = (
    df_final["community_board"]
    .dropna()
    .isin(official_community_boards)
    .eq(False)
    .sum()
)

print(
    "Invalid Community Board values:",
    invalid_community_boards
)

print(
    "PASS - Community Board values are valid"
    if invalid_community_boards == 0
    else "FAIL - Invalid Community Board values detected"
)

# ------------------------------------------------------------
# 8. GEO QUALITY VALIDATION
# ------------------------------------------------------------

print("\n8. GEO QUALITY VALIDATION")
print("-" * 80)

expected_geo_categories = {
    "Coordinates + Borough + Community Board",
    "Coordinates + Borough",
    "Borough + Community Board",
    "Borough + ZIP",
    "Borough Only",
    "Coordinates Only",
    "No Usable Geography"
}

actual_geo_categories = set(
    df_final["geo_quality_final"]
    .dropna()
    .unique()
)

unexpected_geo = (
    actual_geo_categories
    - expected_geo_categories
)

print(
    "Unexpected geographic categories:",
    unexpected_geo
)

print(
    "PASS - Geographic quality categories are valid"
    if len(unexpected_geo) == 0
    else "FAIL - Unexpected geographic categories detected"
)

# ------------------------------------------------------------
# 9. RECORD COUNT
# ------------------------------------------------------------

print("\n9. RECORD COUNT")
print("-" * 80)

print("Final rows:", f"{len(df_final):,}")

print(
    "PASS - Expected 3,655,040 records"
    if len(df_final) == 3_655_040
    else "FAIL - Unexpected record count"
)

# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL ANALYTICAL DATASET VALIDATION COMPLETE")
print("=" * 80)


FINAL ANALYTICAL DATASET VALIDATION

1. COLUMN CHECK
--------------------------------------------------------------------------------
PASS - All required columns are present

2. ID VALIDATION
--------------------------------------------------------------------------------
Unique IDs: True
Missing IDs: 0
PASS - Service request IDs are unique and complete

3. DATE VALIDATION
--------------------------------------------------------------------------------
Missing created_at: 0
PASS - Created dates are complete

4. RESOLUTION VALIDATION
--------------------------------------------------------------------------------
Negative resolution hours: 0
Valid resolution records: 3,525,111
Negative duration records: 893
PASS - No negative values remain in resolution_hours

5. COORDINATE VALIDATION
--------------------------------------------------------------------------------
Invalid latitude values: 0
Invalid longitude values: 0
PASS - Coordinate ranges are valid

6. BOROUGH VALIDATION
-----------

## Conclusion

The recovered 2025 NYC 311 records were validated, standardized, and transformed into a structured analytical dataset for subsequent SQL, geospatial, socioeconomic, and Power BI analysis.
